# 7. Correlación y multicolinealidad

Se usa la **correlación de Spearman**, robusta a la asimetría y a los atípicos observados en el capítulo 4, y el **factor de inflación de la varianza (VIF)**. La multicolinealidad afecta sobre todo a los modelos lineales (regresión logística, Ridge, Lasso).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import *

set_style()
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
df = load_data()

In [ ]:
corr = df[NUMERIC].corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(250, 12, s=75, l=50, center="light", as_cmap=True)

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0, annot=True,
            fmt=".2f", annot_kws={"size": 7.5}, linewidths=1.5, linecolor=SURFACE,
            cbar_kws={"label": "ρ de Spearman", "shrink": 0.7}, ax=ax)
ax.set_title("Correlación de Spearman entre predictoras numéricas")
ax.grid(False)
plt.tight_layout(); plt.show()

In [ ]:
pares = corr.where(~mask).stack().rename("rho").reset_index()
pares.columns = ["variable_1", "variable_2", "rho"]
pares.reindex(pares["rho"].abs().sort_values(ascending=False).index).head(6).round(2)

Hay un único par con correlación fuerte: **`latitude` y `ambient_temperature` (ρ = −0.82)**, es decir, cuanto más lejos del ecuador, más frío.

Le siguen correlaciones moderadas:
- `ghi` con `latitude` (−0.60), `humidity` (−0.58) y `ambient_temperature` (0.53).
- `elevation` con `slope` (0.43).

Las variables climáticas forman un bloque relacionado con la latitud.

## 7.1 Factor de inflación de la varianza

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[NUMERIC].fillna(df[NUMERIC].median())
X = (X - X.mean()) / X.std()
X["const"] = 1.0
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(len(NUMERIC))],
                index=NUMERIC, name="VIF").sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.barh(vif.index[::-1], vif.values[::-1], height=0.6, color=CLASS_COLORS["Alta"])
ax.axvline(5, color=NEUTRAL, lw=0.8, ls=":")
ax.text(5, -0.9, "umbral 5", fontsize=8, color=NEUTRAL, ha="center")
ax.set_xlabel("VIF"); ax.set_title("Multicolinealidad de las predictoras numéricas")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
vif.round(2).to_frame()

Ningún VIF supera 5; el máximo es 3.5, en `ghi`. **No hay multicolinealidad problemática**, así que se pueden conservar las 13 variables numéricas. La regularización L2 de la regresión logística (y L1/L2 en Ridge y Lasso) absorbe sin problema la correlación entre latitud y temperatura.